# 02 · 评级迁移分析

本 notebook 回答四个问题：

1. **年度迁移矩阵**：从某一评级档出发，一年后停留在原档或迁移到其他档的概率是多少？
2. **上调 / 下调概率**：随时间、机构、评级档如何变化？
3. **评级周期**：一个评级平均持续多久？评级行动间隔多久？
4. **机构差异**：同一国家同一年，三家机构的评级分歧有多大？

## 方法要点

* 迁移只统计**相邻年份**（`t+1 == t + 1`）都存在的观测，避免把数据缺口误当作「未迁移」。
* 「无条件概率」（占全部观测）与「条件概率」（占发生变化观测）含义不同，本 notebook 两者都报告。
* 评级档为 1-21，其中 12 为投资级门槛（BBB-/Baa3）。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.analysis.migration import (
    agency_dispersion,
    build_migration_matrix,
    migration_entropy,
    migration_summary,
    rating_cycle_stats,
    rating_spell_stats,
    transition_probability_matrix,
    upgrade_downgrade_probabilities,
)
from src.config import get_path
from src.features.rating_scale import score_to_grade
from src.visualization.plots import plot_migration_heatmap, plot_probability_series, save_figure

PANEL_PATH = get_path("processed") / "panel_country_year.csv"
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, encoding="utf-8-sig")
else:
    from src.clean.panel import build_country_year_panel
    from src.ingest.ratings import load_sample_ratings

    panel = build_country_year_panel(
        load_sample_ratings(),
        pd.read_csv(get_path("sample") / "macro_sample.csv", encoding="utf-8-sig"),
        start_year=2000,
        end_year=2023,
    )
print(f"面板: {panel.shape[0]:,} 行 | 国家 {panel['country_iso3'].nunique()} | 机构 {panel['agency'].nunique()}")

## 1. 全样本迁移矩阵

行 = t 期评级，列 = t+1 期评级。对角线代表「评级未变」。
注意：矩阵很大（21×21），实践中通常也报告**折叠为 7 个大档**的版本。

In [ ]:
counts = build_migration_matrix(panel, score_is_effective=True)
probs = transition_probability_matrix(counts)

summary = migration_summary(counts)
pd.Series(summary).to_frame("数值")

In [ ]:
# 只看样本量足够的行，避免把极小样本的 100% 概率当成结论
row_totals = counts.sum(axis=1)
reliable = probs.loc[row_totals >= 20]
reliable.round(3).style.format("{:.3f}").background_gradient(cmap="Blues", axis=None)

In [ ]:
fig = plot_migration_heatmap(probs, normalize=True, title="主权评级年度迁移概率矩阵")
save_figure(fig, "nb02_migration_matrix")
fig

In [ ]:
# 迁移不确定性：矩阵熵越高，评级体系越不稳定
print(f"全矩阵归一化熵: {migration_entropy(counts):.4f}")
entropy_by_grade = migration_entropy(counts, per_row=True)
entropy_frame = entropy_by_grade.to_frame("熵").join(row_totals.rename("样本量"))
entropy_frame["评级"] = [score_to_grade(i, "S&P") for i in entropy_frame.index]
entropy_frame.loc[row_totals >= 20][["评级", "熵", "样本量"]].round(4)

## 2. 上调 / 下调概率

* `p_upgrade` / `p_downgrade`：占**全部**迁移观测的比例（无条件概率）
* `p_*_given_change`：占**发生变动**观测的比例（条件概率）

两者不可混用。政策讨论中「下调概率」通常指前者。

In [ ]:
by_year = upgrade_downgrade_probabilities(panel, group_cols=["year"])
by_year[["year", "n_transitions", "p_upgrade", "p_downgrade", "p_stable", "mean_delta"]].round(4).tail(15)

In [ ]:
fig = plot_probability_series(
    by_year.set_index("year")[["p_upgrade", "p_downgrade", "p_stable"]],
    title="年度评级迁移概率",
)
save_figure(fig, "nb02_probability_by_year")
fig

In [ ]:
print("—— 按机构 ——")
display(upgrade_downgrade_probabilities(panel, group_cols=["agency"]).round(4))

print("—— 按评级档 ——")
display(upgrade_downgrade_probabilities(panel, group_cols=["bucket_from"]).round(4))

## 3. 评级周期

`rating_spell_stats` 计算每个「国家-机构-评级档」片段的持续年数。
`ongoing=True` 表示该片段在样本期末仍持续（**右删失**，计算平均时长时应剔除或用生存分析处理）。

In [ ]:
spells = rating_spell_stats(panel)
print(f"共 {len(spells)} 个评级片段；右删失占比 {spells['ongoing'].mean():.1%}")
display(spells.sort_values("duration", ascending=False).head(10))

print("—— 按机构 ——")
display(rating_cycle_stats(panel, group_cols=["agency"]).round(3))

## 4. 机构差异（split rating）

同一国家同一年，三家机构给出的评级不完全一致，`range_score > 0` 即存在分歧。
分歧本身是重要的风险信号，也是「不同机构差异」研究的核心度量。

In [ ]:
dispersion = agency_dispersion(panel)
print(f"国家-年份组合: {len(dispersion)} | 存在分歧的比例: {dispersion['is_split'].mean():.1%}")
print(f"平均分歧幅度: {dispersion['range_score'].mean():.3f} 档 | 最大: {dispersion['range_score'].max():.0f} 档")

top_split = dispersion.sort_values(["range_score", "std_score"], ascending=False).head(12)
top_split.round(3)

In [ ]:
split_trend = dispersion.groupby("year").agg(
    分歧比例=("is_split", "mean"), 平均分歧档数=("range_score", "mean")
)
split_trend.round(4)